<a href="https://colab.research.google.com/github/springboardmentor82023/AI_Price_Optima/blob/AI-PRICE-OPTIMA---YUKTHA/demandprice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit pyngrok xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 65.3 MB/s eta 0:00:00


In [2]:
import pandas as pd

df = pd.read_csv("/content/cleaned_retail_inventory.csv")
print(df.columns)
df.head()

Index(['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Price',
       'Units Sold', 'Inventory Level', 'Discount', 'Competitor Pricing',
       'Year', 'Month', 'Day', 'Total Sales'],
      dtype='object')


,Date,Store ID,Product ID,Category,Region,Price,Units Sold,Inventory Level,Discount,Competitor Pricing,Year,Month,Day,Total Sales
0,2022-01-01,S001,P0001,Groceries,North,33.50,127,231,20,29.69,2022,1,1,4254.50
1,2022-01-01,S001,P0002,Toys,South,63.01,150,204,20,66.16,2022,1,1,9451.50
2,2022-01-01,S001,P0003,Toys,West,27.99,65,102,10,31.32,2022,1,1,1819.35
3,2022-01-01,S001,P0004,Toys,North,32.72,61,469,10,34.74,2022,1,1,1995.92
4,2022-01-01,S001,P0005,Electronics,East,73.64,14,166,0,68.95,2022,1,1,1030.96


In [3]:
%%writefile app.py
import streamlit as st
import pandas as pd
import xgboost as xgb

# Load data
df = pd.read_csv("/content/cleaned_retail_inventory.csv")

# Safe column handling
df.columns = df.columns.str.strip()

# If Date exists → use it, else create dummy
if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"])
    df["Month"] = df["Date"].dt.month
    df["Weekday"] = df["Date"].dt.weekday
else:
    df["Month"] = 1
    df["Weekday"] = 1

# Check column names
price_col = "Price"
inventory_col = "Inventory Level"
demand_col = "Units Sold"

# Train model
X = df[[price_col, inventory_col, "Month", "Weekday"]]
y = df[demand_col]

model = xgb.XGBRegressor()
model.fit(X, y)

# UI
st.title("💰 PriceOptima Dashboard")

st.sidebar.header("Input")

price = st.sidebar.slider("Price", 1, 500, 50)
inventory = st.sidebar.slider("Inventory", 1, 200, 50)
month = st.sidebar.slider("Month", 1, 12, 6)
weekday = st.sidebar.slider("Weekday", 0, 6, 3)

# Prediction
input_data = pd.DataFrame({
    price_col: [price],
    inventory_col: [inventory],
    "Month": [month],
    "Weekday": [weekday]
})

predicted_demand = model.predict(input_data)[0]

# Pricing logic
recommended_price = price * 1.1

# Revenue
original_revenue = price * predicted_demand
new_revenue = recommended_price * predicted_demand

lift = ((new_revenue - original_revenue) / original_revenue) * 100

# KPI
st.subheader("📊 Results")

col1, col2, col3, col4 = st.columns(4)

col1.metric("Recommended Price", round(recommended_price,2))
col2.metric("Demand", round(predicted_demand,2))
col3.metric("Revenue", round(new_revenue,2))
col4.metric("Lift %", round(lift,2))

# Chart
st.subheader("📈 Revenue Comparison")

chart = pd.DataFrame({
    "Type": ["Original", "New"],
    "Revenue": [original_revenue, new_revenue]
})

st.bar_chart(chart.set_index("Type"))

# Trend
st.subheader("📉 Demand Trend")
trend = df.groupby("Month")[demand_col].mean()
st.line_chart(trend)

Writing app.py


In [4]:
from pyngrok import conf
conf.get_default().auth_token = "3BiG2Hu6AckfrF4U5WhYHzoDfn6_47ZNg4M2dXo6hSpcWwbhL"

In [5]:
from pyngrok import ngrok
ngrok.kill()

In [6]:
!streamlit run app.py &>/dev/null &

In [7]:
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://marylin-excusive-madonna.ngrok-free.dev" -> "http://localhost:8501"
